# معالج الصور — رفع على BunnyCDN

يأخذ ملف JSON (منتجات + روابط صور)، ينزّل الصور ويرفعها على BunnyCDN، ويرجع ملف النتائج.

**الخطوات:** شغّل الخلايا بالترتيب. في خلية الإعدادات أدناه غيّر مفتاح Bunny ثم شغّل خلية الرفع والمعالجة.

## 1. تثبيت المتطلبات

In [ ]:
!pip install -q httpx nest_asyncio

## 2. الإعدادات (غيّر القيم ثم شغّل الخلية)

In [ ]:
BUNNY_STORAGE_ZONE = "product-listing"
BUNNY_ACCESS_KEY = "ضع_مفتاحك_هنا"
BUNNY_CDN_BASE_URL = "https://product-listing.b-cdn.net"
UPLOAD_PATH_PREFIX = "test"
IMAGE_PATHS = ["Images[]", "variants[].Image"]
PRODUCTS_KEY = ""
PRODUCT_ID_KEY = ""
CONCURRENCY = 20
TIMEOUT = 30
MAX_RETRIES = 3
print("تم ضبط الإعدادات.")

## 3. كود المعالجة (شغّل هذه الخلية مرة واحدة)

In [ ]:
import nest_asyncio
nest_asyncio.apply()  # يسمح بتشغيل asyncio.run() داخل Colab/Jupyter
import asyncio
import hashlib
import json
import mimetypes
import re
from dataclasses import dataclass
from urllib.parse import urlparse
import httpx

REMOVE_MARKER = "__REMOVE__"
SOURCE_URL, SOURCE_FILE, SOURCE_INVALID = "url", "file", "invalid"

def _parse_segments(path_str):
    segs = []
    for part in path_str.split("."):
        if part.endswith("[]"):
            segs.append({"field": part[:-2], "is_array": True})
        else:
            segs.append({"field": part, "is_array": False})
    return segs

class ImageLocation:
    def __init__(self, path_display, keys, original_url, source_type):
        self.path_display, self.keys = path_display, keys
        self.original_url, self.source_type = original_url, source_type

def _is_file_path(s):
    if re.match(r"^[A-Za-z]:[\\/]", s): return True
    if s.startswith(("/", "./", "../", ".\\", "..\\")): return True
    return False

def _classify_source(s):
    if s.startswith(("http://", "https://")): return SOURCE_URL
    if _is_file_path(s): return SOURCE_FILE
    return SOURCE_INVALID

def _format_keys(keys):
    parts = []
    for k in keys:
        if isinstance(k, int): parts[-1] = f"{parts[-1]}[{k}]"
        else: parts.append(k)
    return ".".join(parts)

def _walk(obj, segments, seg_idx, keys_so_far, results):
    if seg_idx >= len(segments):
        if isinstance(obj, str) and obj.strip():
            st = _classify_source(obj.strip())
            results.append(ImageLocation(_format_keys(keys_so_far), list(keys_so_far), obj, st))
        return
    seg = segments[seg_idx]
    field, is_array = seg["field"], seg["is_array"]
    if not isinstance(obj, dict) or field not in obj: return
    value = obj[field]
    if is_array:
        if not isinstance(value, list): return
        for i, item in enumerate(value):
            _walk(item, segments, seg_idx + 1, keys_so_far + [field, i], results)
    else:
        _walk(value, segments, seg_idx + 1, keys_so_far + [field], results)

def extract_image_locations(product, path_str):
    results = []
    _walk(product, _parse_segments(path_str), 0, [], results)
    return results

def update_image_url(product, keys, new_url):
    obj = product
    for key in keys[:-1]: obj = obj[key]
    obj[keys[-1]] = new_url

def cleanup_removed_images(product, path_strs):
    for path_str in path_strs:
        segs = _parse_segments(path_str)
        def _cleanup(obj, seg_idx):
            if seg_idx >= len(segs): return
            f, is_arr = segs[seg_idx]["field"], segs[seg_idx]["is_array"]
            if not isinstance(obj, dict) or f not in obj: return
            v = obj[f]
            if is_arr:
                if not isinstance(v, list): return
                if seg_idx == len(segs) - 1: obj[f] = [x for x in v if x != REMOVE_MARKER]
                else: [ _cleanup(x, seg_idx + 1) for x in v ]
            else:
                if seg_idx == len(segs) - 1 and v == REMOVE_MARKER: obj[f] = None
                else: _cleanup(v, seg_idx + 1)
        _cleanup(product, 0)

@dataclass
class DownloadResult:
    status: str
    data: bytes = None
    content_type: str = ""
    error_message: str = ""
    http_status: int = None
    attempts: int = 0

def guess_ext(ct, data):
    ct = (ct or "").lower()
    for mime, ext in (("image/jpeg","jpg"),("image/png","png"),("image/webp","webp"),("image/gif","gif"),("image/svg+xml","svg"),("image/avif","avif")):
        if mime in ct: return ext
    if data[:3] == b"\xff\xd8\xff": return "jpg"
    if data[:8] == b"\x89PNG\r\n\x1a\n": return "png"
    if data[:4] == b"RIFF" and data[8:12] == b"WEBP": return "webp"
    if data[:6] in (b"GIF87a", b"GIF89a"): return "gif"
    return "jpg"

def generate_filename(data, content_type):
    h = hashlib.sha1(data).hexdigest()[:12]
    return f"{h}.{guess_ext(content_type, data)}"

async def download_image(url, timeout, max_retries):
    ref = f"{urlparse(url).scheme}://{urlparse(url).hostname}/"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/131.0.0.0", "Referer": ref, "Accept": "image/*,*/*;q=0.8"}
    last_err, last_st = "", None
    async with httpx.AsyncClient(timeout=timeout, follow_redirects=True) as client:
        for attempt in range(1, max_retries + 1):
            try:
                r = await client.get(url, headers=headers)
                if r.status_code == 404: return DownloadResult("not_found", http_status=404, attempts=attempt)
                r.raise_for_status()
                return DownloadResult("ok", data=r.content, content_type=r.headers.get("content-type",""), attempts=attempt)
            except httpx.TimeoutException: last_err, last_st = "Request timed out", None
            except httpx.HTTPStatusError as e: last_err, last_st = f"HTTP {e.response.status_code}", e.response.status_code
            except Exception as e: last_err = str(e)
            if attempt < max_retries: await asyncio.sleep(attempt * 2)
    return DownloadResult("failed", error_message=last_err, http_status=last_st, attempts=max_retries)

async def upload_bunny(data, path, filename, zone, key, cdn_base):
    url = f"https://storage.bunnycdn.com/{zone}/{path}/{filename}"
    headers = {"AccessKey": key, "Content-Type": "application/octet-stream"}
    try:
        async with httpx.AsyncClient(timeout=120) as client:
            r = await client.put(url, content=data, headers=headers)
            r.raise_for_status()
        return {"status": "ok", "url": f"{cdn_base.rstrip('/')}/{path}/{filename}" if cdn_base else url}
    except Exception as e:
        return {"status": "failed", "error": str(e)}

def load_json_data(data_str, products_key=""):
    data = json.loads(data_str.decode("utf-8") if isinstance(data_str, bytes) else data_str)
    if isinstance(data, list): return data, data, None
    if isinstance(data, dict):
        if products_key and products_key in data: return data, data[products_key], products_key
        for k, v in data.items():
            if isinstance(v, list) and v and isinstance(v[0], dict): return data, v, k
    raise ValueError("Cannot find products array")

def _sanitize(s): return re.sub(r"[^\\w.-]", "_", str(s)).strip("_")[:100] or "unknown"

def product_id(p, id_key, idx):
    if id_key and id_key in p:
        v = str(p[id_key])
        if v and v.lower() != "nan": return _sanitize(v)
    for key in ("Web SKU", "Amazon ASIN", "sku", "SKU", "id", "asin", "ASIN"):
        if key in p:
            v = str(p[key])
            if v and v.lower() != "nan": return _sanitize(v)
    return f"product_{idx}"

async def process_json(content, image_paths, products_key, product_id_key, upload_prefix, zone, access_key, cdn_base, concurrency, timeout, max_retries):
    raw, products, _ = load_json_data(content, products_key)
    errors = []
    sem = asyncio.Semaphore(concurrency)
    async def do_one(loc, product, pid, idx):
        async with sem:
            if loc.source_type == SOURCE_INVALID:
                errors.append({"product_index": idx, "product_id": pid, "source_url": loc.original_url, "error": "invalid_source"})
                update_image_url(product, loc.keys, REMOVE_MARKER)
                return
            res = await download_image(loc.original_url, timeout, max_retries)
            if res.status == "not_found":
                errors.append({"product_index": idx, "product_id": pid, "source_url": loc.original_url, "error": "not_found"})
                update_image_url(product, loc.keys, REMOVE_MARKER)
                return
            if res.status != "ok":
                errors.append({"product_index": idx, "product_id": pid, "source_url": loc.original_url, "error": res.error_message})
                update_image_url(product, loc.keys, REMOVE_MARKER)
                return
            path = f"{upload_prefix}/{pid}"
            fn = generate_filename(res.data, res.content_type)
            up = await upload_bunny(res.data, path, fn, zone, access_key, cdn_base)
            if up["status"] != "ok":
                errors.append({"product_index": idx, "product_id": pid, "source_url": loc.original_url, "error": up.get("error", "upload failed")})
                update_image_url(product, loc.keys, REMOVE_MARKER)
                return
            update_image_url(product, loc.keys, up["url"])
    tasks = []
    for idx, product in enumerate(products):
        pid = product_id(product, product_id_key, idx)
        for path_str in image_paths:
            for loc in extract_image_locations(product, path_str):
                tasks.append(do_one(loc, product, pid, idx))
    await asyncio.gather(*tasks)
    for product in products:
        cleanup_removed_images(product, image_paths)
    return raw, errors

def run_process(content):
    return asyncio.run(process_json(content, IMAGE_PATHS, PRODUCTS_KEY, PRODUCT_ID_KEY, UPLOAD_PATH_PREFIX,
        BUNNY_STORAGE_ZONE, BUNNY_ACCESS_KEY, BUNNY_CDN_BASE_URL, CONCURRENCY, TIMEOUT, MAX_RETRIES))

print("جاهز. شغّل الخلية التالية لرفع الملف والمعالجة.")

## 4. رفع ملف JSON وتنزيل النتيجة

In [ ]:
from google.colab import files

uploaded = files.upload()
if not uploaded:
    print("لم يتم رفع أي ملف.")
else:
    name = list(uploaded.keys())[0]
    content = uploaded[name]
    print(f"جاري المعالجة: {name} ...")
    result_data, errors = run_process(content)
    base = name.rsplit(".", 1)[0] if "." in name else name
    out_name = f"result_{base}.json"
    err_name = f"errors_{base}.json"
    with open(out_name, "w", encoding="utf-8") as f:
        json.dump(result_data, f, ensure_ascii=False, indent=2)
    with open(err_name, "w", encoding="utf-8") as f:
        json.dump(errors, f, ensure_ascii=False, indent=2)
    print(f"انتهى. {len(errors)} أخطاء.")
    files.download(out_name)
    if errors:
        files.download(err_name)